In [0]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

def load_california_housing(n_samples=None):
    """Fetch the California Housing dataset."""
    dataset = fetch_california_housing(as_frame=True)
    df = dataset.frame
    if n_samples:
        df = df.sample(n_samples, random_state=42)
    return df

# Load data
df = load_california_housing()

# Define target variable
df["PricePerCapita"] = df["MedHouseVal"] / df["Population"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["MedHouseVal", "PricePerCapita"]), 
    df["PricePerCapita"],
    test_size=0.2, 
    random_state=42
)

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# Define preprocessing pipeline
preprocessor = ColumnTransformer([
    ("scaled_numeric", StandardScaler(), ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "Latitude", "Longitude"])
])

# Define the pipeline
pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("regressor", Ridge(alpha=0.5))
        ])

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
import mlflow.pyfunc

# Define preprocessing pipeline
preprocessor = ColumnTransformer([
    ("scaled_numeric", StandardScaler(), ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "Latitude", "Longitude"])
])

def enrich_features(X):
    """Enrich dataset by adding a synthetic 'HousingScore' feature."""
    X = X.copy()
    X["HousingScore"] = (X["MedInc"] * 0.5) + (X["HouseAge"] * 0.3) - (X["AveRooms"] * 0.2)
    return X

class CaliforniaHousingModel(mlflow.pyfunc.PythonModel):
    def __init__(self, alpha=0.5):
        self.alpha = alpha
        self.pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("regressor", Ridge(alpha=alpha))
        ])
    
    def fit(self, X, y):
        """Train the model with enriched data."""
        X_enriched = enrich_features(X)
        self.pipeline.fit(X_enriched, y)

    def predict(self, context, X):
        """Apply feature enrichment before making predictions."""
        X_enriched = enrich_features(X)
        return self.pipeline.predict(X_enriched)

/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:155: FutureWarning: Model's `predict` method contains invalid parameters: {'X'}. Only the following parameter names are allowed: context, model_input, and params. Note that invalid parameters will no longer be permitted in future versions.
  param_names = _check_func_signature(func, "predict")
/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [0]:
# Instantiate and train the custom model
model = CaliforniaHousingModel()
model.fit(X_train, y_train)

In [0]:
mlflow.set_experiment("/Users/linghypshen@gmail.com/ml_experiment")

2026/04/01 21:44:55 INFO mlflow.tracking.fluent: Experiment with name '/Users/linghypshen@gmail.com/ml_experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1470282750680564', creation_time=1775079895761, experiment_id='1470282750680564', last_update_time=1775079895761, lifecycle_stage='active', name='/Users/linghypshen@gmail.com/ml_experiment', tags={'mlflow.experiment.sourceName': '/Users/linghypshen@gmail.com/ml_experiment',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'linghypshen@gmail.com',
 'mlflow.ownerId': '75556308566364'}>

In [0]:
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
print(username)

linghypshen@gmail.com


In [0]:
with mlflow.start_run():
    mlflow.pyfunc.log_model(
        artifact_path="housing_model",
        python_model=model,
        input_example=X_train.iloc[:1],
        registered_model_name="housing_ridge_model"
    )

2026/04/01 21:45:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-de9d8e47-1c1c.cloud.databricks.com/ml/experiments/1470282750680564/models/m-a333eff7ac2b44fd82a7857bdbee2d19?o=1222702474694356
2026/04/01 21:45:33 INFO mlflow.pyfunc: Inferring model signature from input example
Successfully registered model 'workspace.default.housing_ridge_model'.


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.housing_ridge_model': https://dbc-de9d8e47-1c1c.cloud.databricks.com/explore/data/models/workspace/default/housing_ridge_model/version/1?o=1222702474694356


In [0]:
from mlflow.models import infer_signature

hyperparams = [0.1, 0.5, 1.0]
for alpha in hyperparams:
    with mlflow.start_run():
        model = CaliforniaHousingModel(alpha)
        model.fit(X_train, y_train)
        y_pred = model.predict(None, X_test)
        mse = np.mean((y_test - y_pred) ** 2)
        mlflow.log_param("ridge_alpha", alpha)
        mlflow.log_metric("mse", mse)
        
        # Infer signature from training data
        signature = infer_signature(X_train, y_pred)
        
        mlflow.pyfunc.log_model(
            "ridge_model", 
            python_model=model,
            signature=signature,
            input_example=X_train.iloc[:5]
        )

2026/04/01 21:53:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-de9d8e47-1c1c.cloud.databricks.com/ml/experiments/1470282750680564/models/m-b8da2d4f511d467ea97b27f5e9204cfc?o=1222702474694356
2026/04/01 21:53:20 INFO mlflow.pyfunc: Validating input example against model signature
2026/04/01 21:53:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-de9d8e47-1c1c.cloud.databricks.com/ml/experiments/1470282750680564/models/m-3d549be3487d4148ac37172cc65e7b44?o=1222702474694356
2026/04/01 21:53:25 INFO mlflow.pyfunc: Validating input example against model signature
2026/04/01 21:53:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-de9d8e47-1c1c.cloud.databricks.com/ml/experiments/1470282750680564/models/m-00f1c64eecec493c8de3fb5f4f0e0ff7?o=1222702474694356
2026/04/0

In [0]:
RUN_ID = "132a79523777441eb26828b06efdf36e"
model_uri = f"runs:/{RUN_ID}/ridge_model"
mlflow.register_model(model_uri, "best_ridge_model")

Registered model 'best_ridge_model' already exists. Creating a new version of this model...
2026/04/01 21:56:41 WARNING mlflow.tracking._model_registry.fluent: Run with id 132a79523777441eb26828b06efdf36e has no artifacts at artifact path 'ridge_model', registering model based on models:/m-00f1c64eecec493c8de3fb5f4f0e0ff7 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.best_ridge_model': https://dbc-de9d8e47-1c1c.cloud.databricks.com/explore/data/models/workspace/default/best_ridge_model/version/1?o=1222702474694356


<ModelVersion: aliases=[], creation_timestamp=1775080603880, current_stage=None, deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1775080606746, metrics=[<Metric: dataset_digest='', dataset_name='', key='mse', model_id='m-00f1c64eecec493c8de3fb5f4f0e0ff7', run_id='132a79523777441eb26828b06efdf36e', step=0, timestamp=1775080409628, value=0.0001296650001377661>], model_id='m-00f1c64eecec493c8de3fb5f4f0e0ff7', name='workspace.default.best_ridge_model', params=[<LoggedModelParameter: key='ridge_alpha', value='1.0'>], run_id='132a79523777441eb26828b06efdf36e', run_link=None, source='models:/m-00f1c64eecec493c8de3fb5f4f0e0ff7', status='READY', status_message='', tags={}, user_id='linghypshen@gmail.com', version='1'>

In [0]:
from mlflow import MlflowClient

MODEL_NAME = 'workspace.default.best_ridge_model'
client = MlflowClient(registry_uri="databricks-uc")
versions = client.search_model_versions(f"name = '{MODEL_NAME}'")

for v in versions:
    print(
        {
            "name": v.name,
            "version": v.version,
            "current_stage": getattr(v, "current_stage", None),
            "run_id": v.run_id,
            "status": getattr(v, "status", None),
        }
    )

{'name': 'workspace.default.best_ridge_model', 'version': '1', 'current_stage': None, 'run_id': '132a79523777441eb26828b06efdf36e', 'status': 'READY'}


In [0]:
import mlflow.deployments
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")

deploy_client = mlflow.deployments.get_deploy_client("databricks")
registry_client = MlflowClient(registry_uri="databricks-uc")

# Grab the latest version number
latest_version = max(
    int(v.version) for v in registry_client.search_model_versions(f"name = '{MODEL_NAME}'")
)

ENDPOINT_NAME = "iris-rf-demo-endpoint"

endpoint = deploy_client.create_endpoint(
    name=ENDPOINT_NAME,
    config={
        "served_entities": [
            {
                "entity_name": MODEL_NAME,
                "entity_version": str(latest_version),
                "workload_size": "Small",
                "scale_to_zero_enabled": True
            }
        ]
    }
)

print(endpoint)

/databricks/python/lib/python3.12/site-packages/mlflow/deployments/databricks/__init__.py:489: UserWarning: Passing 'name', 'config', and 'route_optimized' as separate parameters is deprecated. Please pass the full API request payload as a single dictionary in the 'config' parameter.
  warnings.warn("\n".join(warnings_list), UserWarning)


{'name': 'iris-rf-demo-endpoint', 'creator': 'linghypshen@gmail.com', 'creation_timestamp': 1775082452000, 'last_updated_timestamp': 1775082452000, 'state': {'ready': 'NOT_READY', 'config_update': 'IN_PROGRESS', 'suspend': 'NOT_SUSPENDED', 'system_update_failure': False}, 'pending_config': {'served_entities': [{'name': 'best_ridge_model-1', 'entity_name': 'workspace.default.best_ridge_model', 'entity_version': '1', 'workload_size': 'Small', 'workload_type': 'CPU', 'scale_to_zero_enabled': True, 'type': 'UC_MODEL', 'state': {'deployment': 'DEPLOYMENT_CREATING', 'deployment_state_message': 'Creating resources for served entity.'}, 'creator': 'linghypshen@gmail.com', 'creation_timestamp': 1775082452000}], 'served_models': [{'name': 'best_ridge_model-1', 'workload_size': 'Small', 'workload_type': 'CPU', 'scale_to_zero_enabled': True, 'model_name': 'workspace.default.best_ridge_model', 'model_version': '1', 'type': 'UC_MODEL', 'state': {'deployment': 'DEPLOYMENT_CREATING', 'deployment_state

In [0]:
import mlflow.deployments

deploy_client = mlflow.deployments.get_deploy_client("databricks")

endpoint_info = deploy_client.get_endpoint(endpoint="iris-rf-demo-endpoint")
print(endpoint_info)

{'name': 'iris-rf-demo-endpoint', 'creator': 'linghypshen@gmail.com', 'creation_timestamp': 1775082452000, 'last_updated_timestamp': 1775082452000, 'state': {'ready': 'READY', 'config_update': 'NOT_UPDATING', 'suspend': 'NOT_SUSPENDED', 'system_update_failure': False}, 'config': {'served_entities': [{'name': 'best_ridge_model-1', 'entity_name': 'workspace.default.best_ridge_model', 'entity_version': '1', 'workload_size': 'Small', 'workload_type': 'CPU', 'scale_to_zero_enabled': True, 'type': 'UC_MODEL', 'state': {'deployment': 'DEPLOYMENT_READY', 'deployment_state_message': ''}, 'creator': 'linghypshen@gmail.com', 'creation_timestamp': 1775082452000}], 'served_models': [{'name': 'best_ridge_model-1', 'workload_size': 'Small', 'workload_type': 'CPU', 'scale_to_zero_enabled': True, 'model_name': 'workspace.default.best_ridge_model', 'model_version': '1', 'type': 'UC_MODEL', 'state': {'deployment': 'DEPLOYMENT_READY', 'deployment_state_message': ''}, 'creator': 'linghypshen@gmail.com', 'c

In [0]:
sample = X_test.head(5)

response = deploy_client.predict(
    endpoint=ENDPOINT_NAME,
    inputs={
        "dataframe_split": sample.to_dict(orient="split")
    }
)

print(response)

{'predictions': [0.0007237739896142951, 0.0023049573469428736, 0.0037335052572827106, 0.0036944064697051436, 0.0038508670675801446]}


In [0]:
import os
import requests

workspace_url = "https://dbc-de9d8e47-1c1c.cloud.databricks.com"
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

url = f"{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}
payload = {
    "dataframe_split": sample.to_dict(orient="split")
}

resp = requests.post(url, headers=headers, json=payload, timeout=60)
print(resp.status_code)
print(resp.json())

200
{'predictions': [0.0007237739896142951, 0.0023049573469428736, 0.0037335052572827106, 0.0036944064697051436, 0.0038508670675801446]}


In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = "workspace.default.best_ridge_model"
MODEL_VERSION = "1"

local_model_dir = mlflow.artifacts.download_artifacts(
    artifact_uri=f"models:/{MODEL_NAME}/{MODEL_VERSION}"
)

print("Downloaded model to:", local_model_dir)

Downloaded model to: /local_disk0/user_tmp_data/spark-adf350a8-ce72-4ba1-8a79-1c/tmp1nin1v73/
